# Phase 2b Stage 2: Entity聚类

对Stage 1.5的626个entities聚类，合并同义词

- 方法: BERTopic（自适应聚类数）
- Canonical name: 按字母顺序选择

In [ ]:
import sys
sys.path.insert(0, '..')

from src.clustering.entity_clusterer import EntityClusterer

## 1. 加载数据

In [ ]:
# 初始化
clusterer = EntityClusterer(
    stage1_5_result_path='../results/entity_redistribution_stage1_5_redistributed.json'
)

# 加载Stage 1.5数据
stage1_5_data = clusterer.load_stage1_5_results()

## 2. 测试单个relation（visual_theme）

In [ ]:
TEST_RELATION = 'visual_theme'

print(f"测试relation: {TEST_RELATION}")
print(f"Entities: {len(clusterer.entities_by_relation[TEST_RELATION])}")

In [ ]:
# 生成embeddings
embeddings, entities = clusterer.embed_entities_bge(TEST_RELATION)

In [ ]:
# BERTopic聚类
topic_model, topics, probs = clusterer.cluster_with_bertopic(
    relation=TEST_RELATION,
    embeddings=embeddings,
    entities=entities,
    min_cluster_size=2,
    verbose=True
)

In [ ]:
# 查看每个topic的详细entities
print(f"\nDetailed topic breakdown for {TEST_RELATION}:")
print("="*80)

for topic_id in sorted(set(topics)):
    topic_entities = [entities[i] for i, t in enumerate(topics) if t == topic_id]
    topic_entities.sort()  # 按字母顺序排序
    
    if topic_id == -1:
        print(f"\nTopic {topic_id} (Noise): {len(topic_entities)} entities")
    else:
        print(f"\nTopic {topic_id}: {len(topic_entities)} entities")
    
    for entity in topic_entities:
        print(f"    {entity}")

## 2.1 参数调优实验

调整BERTopic参数以改善聚类质量（避免大的"垃圾桶"cluster）

**关键参数**:
- `min_cluster_size`: HDBSCAN最小cluster大小（越大越严格）
- `n_neighbors`: UMAP邻居数（影响局部vs全局结构）
- `n_components`: UMAP降维维度
- `min_dist`: UMAP最小距离（控制点的紧密程度）

In [ ]:
# Grid search - 自动测试所有参数组合
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from itertools import product

# 定义参数搜索空间
param_grid = {
    'min_cluster_size': [2, 3, 4, 5],
    'n_neighbors': [5, 10, 15, 20],
    'n_components': [3, 5, 7],
    'min_dist': [0.0, 0.05, 0.1]
}

print(f"总共测试 {len(param_grid['min_cluster_size']) * len(param_grid['n_neighbors']) * len(param_grid['n_components']) * len(param_grid['min_dist'])} 个参数组合...")
print()

results = []

for min_cs, n_neigh, n_comp, min_d in product(
    param_grid['min_cluster_size'],
    param_grid['n_neighbors'],
    param_grid['n_components'],
    param_grid['min_dist']
):
    # UMAP + HDBSCAN + BERTopic
    umap_model = UMAP(
        n_components=n_comp,
        n_neighbors=min(n_neigh, len(entities) - 1),
        min_dist=min_d,
        metric='cosine',
        random_state=42
    )
    
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cs,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        verbose=False,
        calculate_probabilities=False
    )
    
    topics, _ = topic_model.fit_transform(entities, embeddings)
    
    # 统计
    n_clusters = len(set(topics)) - (1 if -1 in topics else 0)
    n_noise = sum(1 for t in topics if t == -1)
    
    # 找最大cluster
    max_cluster_size = 0
    for topic_id in set(topics):
        if topic_id == -1:
            continue
        size = sum(1 for t in topics if t == topic_id)
        if size > max_cluster_size:
            max_cluster_size = size
    
    max_cluster_pct = max_cluster_size / len(entities) if len(entities) > 0 else 0
    
    results.append({
        'min_cluster_size': min_cs,
        'n_neighbors': n_neigh,
        'n_components': n_comp,
        'min_dist': min_d,
        'n_clusters': n_clusters,
        'n_noise': n_noise,
        'max_cluster_size': max_cluster_size,
        'max_cluster_pct': max_cluster_pct
    })

# 排序：优先选择最大cluster百分比小的，其次选择noise少的
results_sorted = sorted(results, key=lambda x: (x['max_cluster_pct'], x['n_noise'] / len(entities)))

print("Top 10 最佳参数组合（按最大cluster百分比排序）:")
print("="*100)
print(f"{'Rank':<5} {'mcs':<4} {'nn':<4} {'nc':<4} {'md':<6} {'#C':<4} {'Noise':<6} {'MaxC':<6} {'MaxC%':<7}")
print("-"*100)

for i, r in enumerate(results_sorted[:10], 1):
    print(f"{i:<5} {r['min_cluster_size']:<4} {r['n_neighbors']:<4} {r['n_components']:<4} "
          f"{r['min_dist']:<6.2f} {r['n_clusters']:<4} {r['n_noise']:<6} "
          f"{r['max_cluster_size']:<6} {r['max_cluster_pct']:<7.1%}")

print("\nmcs=min_cluster_size, nn=n_neighbors, nc=n_components, md=min_dist")
print("#C=总clusters数, MaxC=最大cluster大小, MaxC%=最大cluster百分比")

# 选择最佳参数
best = results_sorted[0]
print(f"\n✨ 推荐参数配置:")
print(f"   min_cluster_size={best['min_cluster_size']}, n_neighbors={best['n_neighbors']}, "
      f"n_components={best['n_components']}, min_dist={best['min_dist']}")
print(f"   结果: {best['n_clusters']} clusters, {best['n_noise']} noise, "
      f"最大cluster {best['max_cluster_size']} ({best['max_cluster_pct']:.1%})")

In [ ]:
# 用最佳参数重新聚类，查看详细结果
best = results_sorted[0]

umap_model_best = UMAP(
    n_components=best['n_components'],
    n_neighbors=min(best['n_neighbors'], len(entities) - 1),
    min_dist=best['min_dist'],
    metric='cosine',
    random_state=42
)

hdbscan_model_best = HDBSCAN(
    min_cluster_size=best['min_cluster_size'],
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

topic_model_best = BERTopic(
    umap_model=umap_model_best,
    hdbscan_model=hdbscan_model_best,
    verbose=False,
    calculate_probabilities=True
)

topics_best, probs_best = topic_model_best.fit_transform(entities, embeddings)

# 详细显示每个topic
print(f"\n最佳参数的详细topic分组:")
print("="*80)

for topic_id in sorted(set(topics_best)):
    topic_entities = [entities[i] for i, t in enumerate(topics_best) if t == topic_id]
    topic_entities.sort()
    
    if topic_id == -1:
        print(f"\nTopic {topic_id} (Noise): {len(topic_entities)} entities")
    else:
        print(f"\nTopic {topic_id}: {len(topic_entities)} entities")
    
    for entity in topic_entities:
        print(f"    {entity}")

In [ ]:
# 生成entity mapping（基于最佳参数）
entity_mapping_test = clusterer.generate_entity_mapping(
    relation=TEST_RELATION,
    topics=topics_best,
    entities=entities
)

# 查看合并的groups
print("\nMerged groups (showing only groups with >1 entity):")
print("="*80)

canonical_groups = {}
for orig, canonical in entity_mapping_test.items():
    if canonical not in canonical_groups:
        canonical_groups[canonical] = []
    canonical_groups[canonical].append(orig)

for canonical in sorted(canonical_groups.keys()):
    group = canonical_groups[canonical]
    if len(group) > 1:
        print(f"\n{canonical}:")
        for ent in sorted(group):
            if ent == canonical:
                print(f"  ★ {ent} [CANONICAL]")
            else:
                print(f"    {ent}")

## 3. 处理所有relations（使用最佳参数）

使用grid search找到的最佳参数:
- min_cluster_size=2
- n_neighbors=20
- n_components=3
- min_dist=0.1

In [ ]:
# 处理所有relations（使用最佳参数）
all_entity_mappings = {}

# 从grid search获得的最佳参数
BEST_PARAMS = {
    'min_cluster_size': 2,
    'n_neighbors': 20,
    'n_components': 3,
    'min_dist': 0.1
}

for idx, relation in enumerate(sorted(clusterer.entities_by_relation.keys()), 1):
    print(f"\n{'='*80}")
    print(f"[{idx}/15] Processing: {relation}")
    print(f"{'='*80}")
    
    n_entities = len(clusterer.entities_by_relation[relation])
    
    # 太少的entity跳过聚类
    if n_entities <= 2:
        print(f"⚠️  Skipping (only {n_entities} entities)")
        all_entity_mappings[relation] = {
            ent: ent for ent in clusterer.entities_by_relation[relation]
        }
        continue
    
    # Embeddings
    embeddings, entities = clusterer.embed_entities_bge(relation)
    
    # BERTopic聚类（使用最佳参数）
    topic_model, topics, probs = clusterer.cluster_with_bertopic(
        relation=relation,
        embeddings=embeddings,
        entities=entities,
        min_cluster_size=BEST_PARAMS['min_cluster_size'],
        n_neighbors=BEST_PARAMS['n_neighbors'],
        n_components=BEST_PARAMS['n_components'],
        min_dist=BEST_PARAMS['min_dist'],
        verbose=False
    )
    
    # 生成mapping
    entity_mapping = clusterer.generate_entity_mapping(relation, topics, entities)
    all_entity_mappings[relation] = entity_mapping

In [ ]:
print(f"\n{'='*80}")
print("Final Statistics")
print(f"{'='*80}")

for relation in sorted(all_entity_mappings.keys()):
    mapping = all_entity_mappings[relation]
    n_before = len(mapping)
    n_after = len(set(mapping.values()))
    compression = n_after / n_before if n_before > 0 else 1.0
    
    print(f"  {relation:25s}: {n_before:3d} → {n_after:3d} ({compression:.1%})")

total_before = sum(len(m) for m in all_entity_mappings.values())
total_after = sum(len(set(m.values())) for m in all_entity_mappings.values())

print(f"\n  {'TOTAL':25s}: {total_before:3d} → {total_after:3d} ({total_after/total_before:.1%})")

## 4. 统计

## 5. 保存结果

In [ ]:
# 保存
clusterer.save_results(
    all_entity_mappings=all_entity_mappings,
    output_path='../results/entity_mapping_bertopic.json',
    metadata={
        'embedding_model': 'BAAI/bge-base-en-v1.5',
        'clustering_method': 'BERTopic',
        'min_cluster_size': 2,
        'n_neighbors': 20,
        'n_components': 3,
        'min_dist': 0.1,
        'canonical_selection': 'alphabetical',
        'note': 'Parameters tuned via grid search on visual_theme relation'
    }
)

print("\n✅ Phase 2b Stage 2 完成！")
print(f"结果已保存到: ../results/entity_mapping_bertopic.json")